<h1 align="center">Pipeline Intermediate Visualizations</h1>

**YOLO detection → SAM2 segmentation → VLM grounding**

This notebook visualizes every intermediate stage of the pipeline and collects statistics:

| Stage | What is shown |
|-------|---------------|
| **YOLO Detection** | Bounding boxes with IDs, class names, confidence scores |
| **Detection Statistics** | Box count, areas, spatial concentration |
| **SAM2 Segmentation** | Colored masks with/without boxes, per-object masks, centroids |
| **Mask Statistics** | Coverage per bounding box |
| **Caption Statistics** | Token / word / sentence counts for VLM captions |

In [ ]:
import json
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from src.sam2 import Sam2Segmenter
from src.utils import draw_detections_cv2, draw_sam2_masks_with_centroids
from src.visualizations import PipelineVisualizer
from src.yolo import YoloV8

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

In [ ]:
def _show(image: np.ndarray, title: str = "", figsize: tuple = (12, 8)) -> None:
    """Displays a single BGR image with an optional title."""

    fig = plt.figure(figsize=figsize)
    if title:
        safe = title.replace("$", r"\$")
        fig.text(0.02, 0.98, safe, va="top", ha="left", fontsize=12, fontweight="bold")
    ax = fig.add_subplot(111)
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.axis("off")
    plt.tight_layout(rect=(0, 0, 1, 0.95) if title else None)
    plt.show()


def _show_pair(images: list, titles: list, figsize: tuple = (20, 8)) -> None:
    """Displays two BGR images side by side."""

    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, t in zip(axes, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def _show_grid(
    images: list, titles: list, cols: int = 3, figsize_per: tuple = (7, 5)
) -> None:
    """Displays BGR images in a grid layout."""

    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(
        rows, cols, figsize=(figsize_per[0] * cols, figsize_per[1] * rows)
    )
    axes_flat = np.array(axes).flatten()
    for ax in axes_flat:
        ax.axis("off")
    for ax, img, t in zip(axes_flat, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
image_dir = project_root / "data" / "processed_images"
image_name = "0004.jpg"  # <-- change this to process a different image
image_path = image_dir / image_name
output_dir = project_root / "outputs"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
original = cv2.imread(str(image_path))
assert original is not None, f"Cannot read image: {image_path}"

yolo_model = YoloV8(device=device)
segmenter = Sam2Segmenter(device=device)

print(f"Device: {device}")
print(f"Image:  {image_name}")

---
## YOLO Detection

YOLOv8 object detection: bounding boxes with numeric IDs, class labels, and confidence scores.

In [ ]:
detections_df = yolo_model.predict(original)
viz = PipelineVisualizer(original, detections_df)
detections = viz.detections

detections_df

### Bounding boxes with IDs

Numbered detection boxes compared with the original image.

In [ ]:
vis_yolo_ids = draw_detections_cv2(original, detections)

_show_pair([original, vis_yolo_ids], ["Original", "YOLO detections (IDs)"])

### Classes and confidence

Each box labeled with its predicted class name and detection confidence.

In [ ]:
vis_yolo_info = viz.draw_detections_with_info()

_show(vis_yolo_info, "YOLO: class names and confidence scores")

### Detection dictionary

Intermediate representation passed to the VLM: `{id: [x1, y1, x2, y2]}`.

In [ ]:
print(json.dumps(VIZ.format_bbox_dict(), indent=2))

### Box statistics

Detection count, average box area, and spatial concentration by image quadrant.

In [ ]:
box_stats = viz.compute_box_statistics()

print(f"Total detections: {box_stats['count']}")
print(f"Mean box area:    {box_stats['mean_area_pct']:.1f}% of image")
print()
print("Spatial distribution (box center quadrant):")
for quadrant, count in box_stats["quadrant_counts"].items():
    print(f"  {quadrant:12s}: {count}")

---
## SAM2 Segmentation

Segment Anything 2 masks generated from YOLO bounding box prompts.

In [ ]:
bboxes = detections["bbox"]
input_boxes = np.array(bboxes, dtype=np.float32)

pil_image = Image.fromarray(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
segmenter.set_image(pil_image)
sam2_out = segmenter.predict_from_boxes(input_boxes, multimask_output=False)
masks = sam2_out.masks

viz.set_masks(masks)

print(f"Generated {len(masks)} masks, shape: {masks.shape}")

### Masks with bounding boxes

SAM2 masks overlaid on the image with bounding box outlines and centroid points (pointing).

In [ ]:
vis_masks_boxes, meta_boxes = viz.draw_colored_masks(boxes=True, draw_centroids=True)

_show(vis_masks_boxes, "SAM2 masks with bounding boxes and centroids")

### Masks without bounding boxes

Same masks but without box outlines — only mask fill, contours, and centroid points.

In [ ]:
vis_masks_only, _ = viz.draw_colored_masks(draw_centroids=True)

_show_pair(
    [vis_masks_boxes, vis_masks_only],
    ["With bounding boxes", "Without bounding boxes"],
)

### Individual object masks

Each detected object shown separately with its colored mask and centroid.

In [ ]:
n_objects = len(masks) if masks.ndim >= 3 else 1
mask_labels = [f"Object {i + 1}: {detections['class'][i]}" for i in range(n_objects)]
mask_images = [viz.draw_single_object_mask(i) for i in range(n_objects)]

_show_grid(mask_images, mask_labels, cols=min(n_objects, 4))

### Mask coverage statistics

How much of each bounding box is actually occupied by the segmentation mask (in percent).

In [ ]:
coverage = viz.compute_mask_coverage()

print(f"Mean mask coverage: {coverage['mean_coverage_pct']}% of bounding box\n")
for obj in coverage["per_object"]:
    print(
        f"  Object {obj['id']:2d}: {obj['coverage_pct']:5.1f}%"
        f"  (mask {obj['mask_area_px']:,}px / box {obj['box_area_px']:,}px)"
    )

---
## Caption Statistics

Token and word counts for the VLM-generated caption (loaded from saved outputs).

In [ ]:
caption_path = output_dir / "captions" / f"{image_path.stem}.json"

if caption_path.exists():
    with open(caption_path) as f:
        caption_data = json.load(f)
    caption_text = str(caption_data.get("answer", ""))
    caption_stats = PipelineVisualizer.compute_caption_stats(caption_text)

    print(f"Caption: {caption_text[:120]}...\n")
    print(f"  Characters:     {caption_stats['char_count']}")
    print(f"  Words:          {caption_stats['word_count']}")
    print(f"  Sentences:      {caption_stats['sentence_count']}")
    print(f"  Approx. tokens: {caption_stats['approx_token_count']}")
else:
    print(f"No caption output found: {caption_path}")

---
## Pipeline Summary

All intermediate stages at a glance.

In [ ]:
vis_pipeline, _ = draw_sam2_masks_with_centroids(
    original,
    masks,
    alpha=0.45,
    circle_radius=8,
    text_scale=1.0,
    text_thickness=3,
    start_index=1,
)

_show_grid(
    images=[
        original,
        vis_yolo_info,
        vis_yolo_ids,
        vis_masks_boxes,
        vis_masks_only,
        vis_pipeline,
    ],
    titles=[
        "Original",
        "YOLO (class + confidence)",
        "YOLO (numbered IDs)",
        "SAM2 masks + boxes",
        "SAM2 masks only",
        "Pipeline output (centroids)",
    ],
    cols=3,
)